# 01 - Exploratory Data Analysis (EDA)

Este notebook realiza un análisis exploratorio inicial del dataset sintético de churn en un entorno SaaS tipo martech.

Objetivos:
- revisar la estructura general de los datos,
- validar la distribución de la variable objetivo,
- explorar patrones de churn por segmento y variables clave,
- identificar señales relevantes para el modelado posterior,
- generar tablas y gráficos reutilizables en el TFM.


## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


## 2. Carga del dataset

Ajusta la ruta si el archivo CSV se encuentra en otra carpeta.


In [ ]:
file_path = 'synthetic_saas_churn_dataset.csv'
df = pd.read_csv(file_path)

print('Shape del dataset:', df.shape)
df.head()


## 3. Revisión general de la estructura

In [ ]:
df.info()

In [ ]:
df.describe(include='all').transpose()

## 4. Comprobación de valores nulos y duplicados

In [ ]:
nulls = df.isnull().sum().sort_values(ascending=False)
print('Valores nulos por columna:')
print(nulls[nulls > 0])

print('\nNúmero de account_id duplicados:', df['account_id'].duplicated().sum())


## 5. Distribución de la variable objetivo

In [ ]:
churn_dist = df['churn_label'].value_counts().sort_index()
churn_pct = df['churn_label'].value_counts(normalize=True).sort_index() * 100

summary_churn = pd.DataFrame({
    'count': churn_dist,
    'percentage': churn_pct.round(2)
})
summary_churn.index = ['No churn (0)', 'Churn (1)']
summary_churn


In [ ]:
plt.figure(figsize=(6,4))
churn_dist.plot(kind='bar')
plt.title('Distribución de churn_label')
plt.xlabel('Clase')
plt.ylabel('Número de cuentas')
plt.xticks([0,1], ['0 = No churn', '1 = Churn'], rotation=0)
plt.tight_layout()
plt.show()


## 6. Métricas de negocio agregadas

In [ ]:
churn_rate = df['churn_label'].mean()
retention_rate = ((df['renewed_last_quarter'] == 1) | (df['subscription_status'] == 'active')).mean()

business_metrics = pd.DataFrame({
    'metric': ['Quarterly churn rate', 'Quarterly retention rate'],
    'value': [round(churn_rate * 100, 2), round(retention_rate * 100, 2)]
})
business_metrics


## 7. Distribución por segmento, tipo de suscripción y plan

In [ ]:
for col in ['segment', 'subscription_type', 'plan_tier', 'renewal_intent', 'subscription_status']:
    print(f'\n===== {col} =====')
    print(df[col].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['segment'].value_counts().plot(kind='bar', ax=axes[0], title='Segment')
df['subscription_type'].value_counts().plot(kind='bar', ax=axes[1], title='Subscription type')
df['plan_tier'].value_counts().plot(kind='bar', ax=axes[2], title='Plan tier')

for ax in axes:
    ax.set_xlabel('')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()


## 8. Tasa de churn por categorías relevantes

In [ ]:
def churn_by_category(df, col):
    result = df.groupby(col)['churn_label'].agg(['count', 'mean']).sort_values('mean', ascending=False)
    result['churn_rate_pct'] = (result['mean'] * 100).round(2)
    return result[['count', 'churn_rate_pct']]

for col in ['segment', 'subscription_type', 'plan_tier', 'renewal_intent', 'subscription_status']:
    print(f'\n===== Churn por {col} =====')
    print(churn_by_category(df, col))


In [ ]:
churn_segment = df.groupby('segment')['churn_label'].mean().sort_values(ascending=False) * 100

plt.figure(figsize=(6,4))
churn_segment.plot(kind='bar')
plt.title('Churn rate por segmento')
plt.ylabel('Churn rate (%)')
plt.xlabel('Segmento')
plt.tight_layout()
plt.show()


## 9. Comparación de variables numéricas entre churn y no churn

In [ ]:
numeric_cols = [
    'tenure_months', 'mrr', 'bundles_contracted', 'extra_users', 'renewal_due_days',
    'login_count_30d', 'tool_activity_score', 'usage_change_vs_prev_quarter',
    'feature_adoption_score', 'key_reports_used_30d', 'days_since_last_login',
    'emails_with_csm_30d', 'csm_meetings_last_quarter', 'tickets_last_quarter',
    'reopened_tickets', 'avg_resolution_days', 'csat_support',
    'sentiment_csm', 'sentiment_support'
]

comparison = df.groupby('churn_label')[numeric_cols].mean().transpose()
comparison.columns = ['No churn (0)', 'Churn (1)']
comparison['Difference'] = comparison['Churn (1)'] - comparison['No churn (0)']
comparison.sort_values('Difference')


## 10. Visualizaciones clave para el TFM

In [ ]:
key_metrics = [
    'tool_activity_score',
    'feature_adoption_score',
    'usage_change_vs_prev_quarter',
    'csat_support',
    'sentiment_csm',
    'reopened_tickets'
]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(key_metrics):
    means = df.groupby('churn_label')[col].mean()
    means.index = ['No churn', 'Churn']
    means.plot(kind='bar', ax=axes[i], title=col)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Average value')

plt.tight_layout()
plt.show()


## 11. Correlación entre variables numéricas

La correlación no implica causalidad, pero ayuda a identificar señales que parecen moverse junto al churn.


In [ ]:
corr_cols = numeric_cols + ['churn_label']
corr = df[corr_cols].corr(numeric_only=True)['churn_label'].sort_values(ascending=False)
corr


In [ ]:
plt.figure(figsize=(8,6))
corr.drop('churn_label').sort_values().plot(kind='barh')
plt.title('Correlación de variables numéricas con churn_label')
plt.xlabel('Correlación')
plt.tight_layout()
plt.show()


## 12. Segmentación simple de cuentas de riesgo

Esta parte no sustituye al modelo predictivo, pero ayuda a observar patrones tempranos.


In [ ]:
risk_snapshot = df.copy()

risk_snapshot['low_activity_flag'] = (risk_snapshot['tool_activity_score'] < 35).astype(int)
risk_snapshot['low_adoption_flag'] = (risk_snapshot['feature_adoption_score'] < 40).astype(int)
risk_snapshot['negative_sentiment_flag'] = (risk_snapshot['sentiment_csm'] < -0.2).astype(int)
risk_snapshot['poor_support_flag'] = (risk_snapshot['csat_support'] < 3.2).astype(int)
risk_snapshot['renewal_risk_flag'] = ((risk_snapshot['renewal_due_days'] < 30) & (risk_snapshot['renewal_intent'] == 'negative')).astype(int)

risk_snapshot['simple_risk_signals'] = risk_snapshot[
    ['low_activity_flag', 'low_adoption_flag', 'negative_sentiment_flag', 'poor_support_flag', 'renewal_risk_flag']
].sum(axis=1)

risk_snapshot.groupby('simple_risk_signals')['churn_label'].agg(['count', 'mean'])


## 13. Tabla resumen para usar en la memoria

In [ ]:
summary_table = pd.DataFrame({
    'metric': [
        'Total accounts',
        'Quarterly churn rate (%)',
        'Quarterly retention rate (%)',
        'Average activity score',
        'Average feature adoption score',
        'Average CSAT support',
        'Average sentiment with CSM'
    ],
    'value': [
        len(df),
        round(df['churn_label'].mean() * 100, 2),
        round((((df['renewed_last_quarter'] == 1) | (df['subscription_status'] == 'active')).mean()) * 100, 2),
        round(df['tool_activity_score'].mean(), 2),
        round(df['feature_adoption_score'].mean(), 2),
        round(df['csat_support'].mean(), 2),
        round(df['sentiment_csm'].mean(), 2)
    ]
})

summary_table


## 14. Conclusiones iniciales del EDA

Puntos que puedes adaptar luego a la memoria:

- El dataset presenta una estructura coherente para abordar un problema de churn en un entorno SaaS.
- La variable objetivo muestra una distribución plausible y no completamente equilibrada, lo que se asemeja a un escenario real.
- Existen diferencias visibles entre cuentas churned y no churned en variables como actividad, adopción, satisfacción y sentimiento.
- Las señales relacionadas con uso reciente, renovación, sentimiento y experiencia de soporte parecen especialmente relevantes para el modelado.
- El análisis exploratorio valida que el problema puede abordarse mediante modelos predictivos en la siguiente fase del proyecto.
